In [0]:
MY_ID = "preetam"

In [0]:
# STEP 3 — Silver (the four decisions, in order)

import pyspark.sql.functions as F
from pyspark.sql import Window as W

m = spark.table(f"workspace.capstone_{MY_ID}.bronze_marks")

# 1. CAST with try_cast, AB -> 0
m1 = m.selectExpr(
    "student_id", "semester", "subject_code",
    "try_cast(classes_held as int) classes_held",
    "try_cast(classes_attended as int) classes_attended",
    "coalesce(try_cast(internal_marks as int), 0) internal_marks",
    "try_cast(external_marks as int) external_marks",
    "_ingested_at"
).withColumn("total_marks", F.col("internal_marks") + F.col("external_marks"))

# sanity: exactly 300 internal_marks originally null before coalesce
assert m.filter("internal_marks = 'AB'").count() == 300

# 2. DEDUPE on (student_id, semester, subject_code), then reject bad rows
w = W.partitionBy("student_id", "semester", "subject_code").orderBy("_ingested_at")
m2 = m1.withColumn("rn", F.row_number().over(w)).filter("rn = 1").drop("rn")
print("after dedupe:", m2.count())          # expect 21,720

students = spark.table(f"workspace.capstone_{MY_ID}.bronze_students").select("student_id").distinct()

rejects = []
m3 = m2.join(students, "student_id", "left_semi")            # drop unknown_student
rej_unknown = m2.join(students, "student_id", "left_anti").withColumn("reason", F.lit("unknown_student"))
print("after unknown_student reject:", m3.count())            # expect 21,600

m4 = m3.filter("classes_attended <= classes_held")
rej_attend = m3.filter("classes_attended > classes_held").withColumn("reason", F.lit("attendance_over_100"))
m5 = m4.filter("classes_held != 0")
rej_zero = m4.filter("classes_held = 0").withColumn("reason", F.lit("no_classes_held"))
m6 = m5.filter("external_marks <= 60")
rej_ext = m5.filter("external_marks > 60").withColumn("reason", F.lit("marks_over_max"))
print("silver rows:", m6.count())                              # expect 21,160

silver_rejects = rej_unknown.unionByName(rej_attend, allowMissingColumns=True) \
    .unionByName(rej_zero, allowMissingColumns=True) \
    .unionByName(rej_ext, allowMissingColumns=True)

# 3. ATTENDANCE band, on the UNROUNDED percentage
m7 = m6.withColumn("attendance_pct", F.col("classes_attended") / F.col("classes_held") * 100) \
       .withColumn("attendance_band",
            F.when(F.col("attendance_pct") < 60, "BELOW_60")
             .when(F.col("attendance_pct") < 75, "BAND_60_74")
             .when(F.col("attendance_pct") < 85, "BAND_75_84")
             .otherwise("BAND_85_100"))

# 4. SEM TOTAL, IS_COMPLETE, RANK — only over students with all 6 subjects surviving
cnt_w = W.partitionBy("student_id", "semester")
m8 = m7.withColumn("subjects_in_sem", F.count("*").over(cnt_w)) \
       .withColumn("sem_total_marks", F.sum("total_marks").over(cnt_w)) \
       .withColumn("is_complete", F.col("subjects_in_sem") == 6) \
       .drop("subjects_in_sem")

rank_w = W.partitionBy("semester").orderBy(F.desc("sem_total_marks"))
complete = m8.filter("is_complete = true").select("student_id","semester","sem_total_marks").distinct() \
    .withColumn("sem_rank", F.rank().over(rank_w))

silver = m8.join(complete.select("student_id","semester","sem_rank"), ["student_id","semester"], "left")

silver.write.mode("overwrite").saveAsTable(f"workspace.capstone_{MY_ID}.silver_marks")
silver_rejects.write.mode("overwrite").saveAsTable(f"workspace.capstone_{MY_ID}.silver_rejects")

after dedupe: 21720
after unknown_student reject: 21600
silver rows: 21160
